In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

results = pd.read_csv('results.csv')
elo = pd.read_csv('elo_ratings_wc2026.csv')
fixtures = pd.read_csv('wc_2026_fixtures.csv')
teams = pd.read_csv('wc_2026_teams.csv')

print("=== RESULTS ===")
print(results.shape)
print(results.head(3))

print("\n=== ELO ===")
print(elo.shape)
print(elo.head(3))

print("\n=== FIXTURES ===")
print(fixtures.shape)
print(fixtures.head(3))

print("\n=== TEAMS ===")
print(teams.shape)
print(teams.head(3))

=== RESULTS ===
(49477, 9)
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  

=== ELO ===
(4683, 23)
   year snapshot_date    country  rank country_code  rating  rank_max  \
0  1901    1901-12-31    England     1           EN    2013         1   
1  1901    1901-12-31   Scotland     2           SQ    1973         1   
2  1902    1902-12-31  Argentina     1           AR    2021         1   

   rating_max  rank_avg  rating_avg  ...  matches_home  matches_away  \
0        2079         2        1989  ...            38            35   
1        2104         1        2018  ...            37            37   
2        2021         1

In [ ]:
# Convertir la date
results['date'] = pd.to_datetime(results['date'])

# Filtrer depuis 2000
results = results[results['date'] >= '2000-01-01']

print(results['tournament'].value_counts().head(20))

tournament
Friendly                                8448
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
AFC Asian Cup qualification              530
African Cup of Nations                   525
FIFA World Cup                           456
CONCACAF Nations League                  422
Gold Cup                                 359
CECAFA Cup                               341
COSAFA Cup                               326
CFU Caribbean Cup qualification          293
Island Games                             283
UEFA Euro                                277
AFC Asian Cup                            256
AFF Championship                         251
Copa América                             248
Gulf Cup                                 189
SAFF Cup                                 135
Name: count, dtype: int64


In [ ]:
official_tournaments = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa América',
    'African Cup of Nations',
    'African Cup of Nations qualification',
    'UEFA Nations League',
    'Gold Cup',
    'AFC Asian Cup',
    'AFC Asian Cup qualification',
    'CONCACAF Nations League'
]

results_filtered = results[results['tournament'].isin(official_tournaments)]

print(f"Avant : {len(results)} matchs")
print(f"Après : {len(results_filtered)} matchs")
print(results_filtered['tournament'].value_counts())

Avant : 25415 matchs
Après : 12519 matchs
tournament
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
AFC Asian Cup qualification              530
African Cup of Nations                   525
FIFA World Cup                           456
CONCACAF Nations League                  422
Gold Cup                                 359
UEFA Euro                                277
AFC Asian Cup                            256
Copa América                             248
Name: count, dtype: int64


In [ ]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

results_filtered = results_filtered.copy()
results_filtered['result'] = results_filtered.apply(get_result, axis=1)

print(results_filtered['result'].value_counts())

result
home_win    6058
away_win    3679
draw        2782
Name: count, dtype: int64


In [ ]:
# Garder seulement le dernier snapshot ELO par équipe
elo_latest = elo.sort_values('year').groupby('country').last().reset_index()

# Garder seulement les colonnes utiles
elo_latest = elo_latest[['country', 'rating', 'confederation']]

print(elo_latest.shape)
print(elo_latest.head(10))

(48, 3)
                  country  rating confederation
0                 Algeria    1743           CAF
1               Argentina    2113      CONMEBOL
2               Australia    1783           AFC
3                 Austria    1827          UEFA
4                 Belgium    1867          UEFA
5  Bosnia and Herzegovina    1594          UEFA
6                  Brazil    1984      CONMEBOL
7                  Canada    1784      CONCACAF
8              Cape Verde    1549           CAF
9                Colombia    1975      CONMEBOL


In [ ]:
# Merger ELO pour home team
results_filtered = results_filtered.merge(
    elo_latest[['country', 'rating']].rename(columns={'country': 'home_team', 'rating': 'home_elo'}),
    on='home_team', how='left'
)

# Merger ELO pour away team
results_filtered = results_filtered.merge(
    elo_latest[['country', 'rating']].rename(columns={'country': 'away_team', 'rating': 'away_elo'}),
    on='away_team', how='left'
)

print(results_filtered[['home_team', 'home_elo', 'away_team', 'away_elo', 'result']].head(10))
print(f"\nNaN home_elo: {results_filtered['home_elo'].isna().sum()}")
print(f"NaN away_elo: {results_filtered['away_elo'].isna().sum()}")

      home_team  home_elo    away_team  away_elo    result
0         Ghana    1503.0     Cameroon       NaN      draw
1         China       NaN  Philippines       NaN  home_win
2         Egypt    1689.0       Zambia       NaN  home_win
3       Nigeria       NaN      Tunisia    1636.0  home_win
4  South Africa    1524.0        Gabon       NaN  home_win
5       Vietnam       NaN         Guam       NaN  home_win
6      DR Congo    1655.0      Algeria    1743.0      draw
7   Ivory Coast    1676.0         Togo       NaN      draw
8  Burkina Faso       NaN      Senegal    1878.0  away_win
9       Morocco    1822.0        Congo       NaN  home_win

NaN home_elo: 8239
NaN away_elo: 8535


In [ ]:
results_clean = results_filtered.dropna(subset=['home_elo', 'away_elo'])

print(f"Matchs restants : {len(results_clean)}")
print(results_clean['result'].value_counts())

Matchs restants : 1596
result
home_win    710
draw        472
away_win    414
Name: count, dtype: int64


In [ ]:
# Reset — repartir de results_filtered sans ELO
results_filtered = results_filtered.drop(columns=['home_elo', 'away_elo'])

# Ajouter l'année dans results
results_filtered['year'] = results_filtered['date'].dt.year

# ELO par équipe par année
elo_by_year = elo[['country', 'year', 'rating']].copy()

# Merger home team
results_filtered = results_filtered.merge(
    elo_by_year.rename(columns={'country': 'home_team', 'rating': 'home_elo'}),
    on=['home_team', 'year'], how='left'
)

# Merger away team
results_filtered = results_filtered.merge(
    elo_by_year.rename(columns={'country': 'away_team', 'rating': 'away_elo'}),
    on=['away_team', 'year'], how='left'
)

print(f"NaN home_elo: {results_filtered['home_elo'].isna().sum()}")
print(f"NaN away_elo: {results_filtered['away_elo'].isna().sum()}")
print(f"Total matchs : {len(results_filtered)}")

NaN home_elo: 8245
NaN away_elo: 8551
Total matchs : 12760


In [ ]:
results_filtered = results_filtered[results_filtered['date'] >= '2010-01-01']

print(f"Matchs depuis 2010 : {len(results_filtered)}")
print(results_filtered['tournament'].value_counts())

Matchs depuis 2010 : 8518
tournament
FIFA World Cup qualification            3430
African Cup of Nations qualification    1052
UEFA Euro qualification                 1016
UEFA Nations League                      658
FIFA World Cup                           538
CONCACAF Nations League                  422
African Cup of Nations                   388
AFC Asian Cup qualification              269
Gold Cup                                 225
UEFA Euro                                184
Copa América                             170
AFC Asian Cup                            166
Name: count, dtype: int64


In [ ]:
results_filtered = results[results['date'] >= '2006-01-01']
results_filtered = results_filtered[results_filtered['tournament'].isin(official_tournaments)]

print(f"Matchs depuis 2006 : {len(results_filtered)}")

Matchs depuis 2006 : 9891


In [ ]:
# Target variable
results_filtered = results_filtered.copy()
results_filtered['result'] = results_filtered.apply(get_result, axis=1)
results_filtered['year'] = results_filtered['date'].dt.year

# ELO par année
elo_by_year = elo[['country', 'year', 'rating']].copy()

# Merger home ELO
results_filtered = results_filtered.merge(
    elo_by_year.rename(columns={'country': 'home_team', 'rating': 'home_elo'}),
    on=['home_team', 'year'], how='left'
)

# Merger away ELO
results_filtered = results_filtered.merge(
    elo_by_year.rename(columns={'country': 'away_team', 'rating': 'away_elo'}),
    on=['away_team', 'year'], how='left'
)

print(f"NaN home_elo: {results_filtered['home_elo'].isna().sum()}")
print(f"NaN away_elo: {results_filtered['away_elo'].isna().sum()}")
print(f"Total matchs : {len(results_filtered)}")

NaN home_elo: 6544
NaN away_elo: 6774
Total matchs : 10132


In [ ]:
print(f"Années disponibles dans ELO : {elo['year'].min()} → {elo['year'].max()}")
print(f"Nombre d'équipes uniques : {elo['country'].nunique()}")
print(elo['year'].value_counts().sort_index().tail(10))

Années disponibles dans ELO : 1901 → 2026
Nombre d'équipes uniques : 48
year
2017    48
2018    48
2019    48
2020    48
2021    48
2022    48
2023    48
2024    48
2025    48
2026    96
Name: count, dtype: int64


In [ ]:
# Liste des 48 équipes qualifiées
wc_teams = teams['team'].tolist()
print(f"Nombre d'équipes : {len(wc_teams)}")
print(wc_teams)

Nombre d'équipes : 48
['Mexico', 'South Africa', 'South Korea', 'Czechia', 'Canada', 'Switzerland', 'Qatar', 'Bosnia and Herzegovina', 'Brazil', 'Morocco', 'Haiti', 'Scotland', 'USA', 'Paraguay', 'Australia', 'Türkiye', 'Germany', 'Curaçao', 'Ivory Coast', 'Ecuador', 'Netherlands', 'Japan', 'Sweden', 'Tunisia', 'Belgium', 'Egypt', 'Iran', 'New Zealand', 'Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay', 'France', 'Senegal', 'Norway', 'Iraq', 'Argentina', 'Algeria', 'Austria', 'Jordan', 'Portugal', 'DR Congo', 'Uzbekistan', 'Colombia', 'England', 'Croatia', 'Ghana', 'Panama']


In [ ]:
wc_teams_official = [
    # Hosts
    'Canada', 'Mexico', 'USA',
    # AFC
    'Australia', 'Iraq', 'Iran', 'Japan', 'Jordan',
    'South Korea', 'Qatar', 'Saudi Arabia', 'Uzbekistan',
    # CAF
    'Algeria', 'Cape Verde', 'DR Congo', 'Ivory Coast',
    'Egypt', 'Ghana', 'Morocco', 'Senegal', 'South Africa', 'Tunisia',
    # CONCACAF
    'Curaçao', 'Haiti', 'Panama',
    # CONMEBOL
    'Argentina', 'Brazil', 'Colombia', 'Ecuador', 'Paraguay', 'Uruguay',
    # OFC
    'New Zealand',
    # UEFA
    'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Croatia',
    'Czechia', 'England', 'France', 'Germany', 'Netherlands',
    'Norway', 'Portugal', 'Scotland', 'Spain', 'Sweden',
    'Switzerland', 'Türkiye'
]

print(f"Total équipes : {len(wc_teams_official)}")

Total équipes : 48


In [ ]:
# Filtrer seulement les matchs entre équipes du WC 2026
results_wc = results_filtered[
    (results_filtered['home_team'].isin(wc_teams_official)) &
    (results_filtered['away_team'].isin(wc_teams_official))
].copy()

print(f"Matchs entre équipes WC 2026 : {len(results_wc)}")
print(results_wc['result'].value_counts())

Matchs entre équipes WC 2026 : 1333
result
draw        521
home_win    510
away_win    302
Name: count, dtype: int64


In [ ]:
# Stats de base
print(f"Total matchs pour training : {len(results_filtered)}")
print(f"Equipes uniques : {results_filtered['home_team'].nunique()}")

# Vérifier que nos 48 équipes sont bien présentes
teams_in_data = set(results_filtered['home_team'].tolist() + results_filtered['away_team'].tolist())
missing = [t for t in wc_teams_official if t not in teams_in_data]
print(f"\nEquipes WC manquantes dans la data : {missing}")

Total matchs pour training : 10132
Equipes uniques : 217

Equipes WC manquantes dans la data : ['USA', 'Czechia', 'Türkiye']


In [ ]:
# Renommer dans results_filtered
name_mapping = {
    'United States': 'USA',
    'Czech Republic': 'Czechia',
    'Turkey': 'Türkiye'
}

results_filtered['home_team'] = results_filtered['home_team'].replace(name_mapping)
results_filtered['away_team'] = results_filtered['away_team'].replace(name_mapping)

# Vérifier
teams_in_data = set(results_filtered['home_team'].tolist() + results_filtered['away_team'].tolist())
missing = [t for t in wc_teams_official if t not in teams_in_data]
print(f"Equipes WC manquantes : {missing}")

Equipes WC manquantes : []


In [ ]:
def get_team_stats(team, date, df, n=10):
    # Matchs de cette équipe avant cette date
    matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)

    if len(matches) == 0:
        return {'win_rate': 0.5, 'avg_goals_scored': 1.0, 'avg_goals_conceded': 1.0}

    wins, goals_scored, goals_conceded = 0, 0, 0

    for _, row in matches.iterrows():
        if row['home_team'] == team:
            goals_scored += row['home_score']
            goals_conceded += row['away_score']
            if row['result'] == 'home_win':
                wins += 1
        else:
            goals_scored += row['away_score']
            goals_conceded += row['home_score']
            if row['result'] == 'away_win':
                wins += 1

    n_matches = len(matches)
    return {
        'win_rate': wins / n_matches,
        'avg_goals_scored': goals_scored / n_matches,
        'avg_goals_conceded': goals_conceded / n_matches
    }

# Test sur une équipe
print(get_team_stats('France', '2026-01-01', results_filtered))

{'win_rate': 0.7, 'avg_goals_scored': 2.4, 'avg_goals_conceded': 1.1}


In [ ]:
from tqdm import tqdm

features = []

for _, row in tqdm(results_filtered.iterrows(), total=len(results_filtered)):
    home_stats = get_team_stats(row['home_team'], row['date'], results_filtered)
    away_stats = get_team_stats(row['away_team'], row['date'], results_filtered)

    features.append({
        'home_win_rate': home_stats['win_rate'],
        'home_goals_scored': home_stats['avg_goals_scored'],
        'home_goals_conceded': home_stats['avg_goals_conceded'],
        'away_win_rate': away_stats['win_rate'],
        'away_goals_scored': away_stats['avg_goals_scored'],
        'away_goals_conceded': away_stats['avg_goals_conceded'],
        'result': row['result']
    })

df_features = pd.DataFrame(features)
print(df_features.shape)
print(df_features.head())

100%|██████████| 10132/10132 [01:21<00:00, 125.08it/s]

(10132, 7)
   home_win_rate  home_goals_scored  home_goals_conceded  away_win_rate  \
0            0.5                1.0                  1.0            0.5   
1            0.5                1.0                  1.0            0.5   
2            0.5                1.0                  1.0            0.5   
3            0.5                1.0                  1.0            0.5   
4            0.5                1.0                  1.0            0.5   

   away_goals_scored  away_goals_conceded    result  
0                1.0                  1.0  home_win  
1                1.0                  1.0  home_win  
2                1.0                  1.0  away_win  
3                1.0                  1.0  away_win  
4                1.0                  1.0  away_win  


In [ ]:
print(df_features.iloc[5000:5005])
print("\n")
print(df_features.tail())

      home_win_rate  home_goals_scored  home_goals_conceded  away_win_rate  \
5000            0.4                1.9                  2.0            0.2   
5001            0.1                0.5                  2.2            0.6   
5002            0.4                1.2                  1.4            0.2   
5003            0.4                1.6                  1.3            0.3   
5004            0.4                1.3                  1.1            0.2   

      away_goals_scored  away_goals_conceded    result  
5000                1.0                  2.1  home_win  
5001                1.9                  0.8  home_win  
5002                0.7                  1.2  home_win  
5003                0.7                  1.8  home_win  
5004                0.5                  3.1  home_win  


       home_win_rate  home_goals_scored  home_goals_conceded  away_win_rate  \
10127            0.2                NaN                  NaN            0.2   
10128            0.2         

In [ ]:
# Ajouter ELO difference comme feature
elo_latest = elo[elo['year'] == 2025][['country', 'rating']].copy()

# Merger ELO dans results_filtered
results_filtered = results_filtered.drop(columns=['home_elo', 'away_elo'], errors='ignore')

results_filtered = results_filtered.merge(
    elo_latest.rename(columns={'country': 'home_team', 'rating': 'home_elo'}),
    on='home_team', how='left'
)
results_filtered = results_filtered.merge(
    elo_latest.rename(columns={'country': 'away_team', 'rating': 'away_elo'}),
    on='away_team', how='left'
)

# Ajouter ELO dans df_features
df_features['home_elo'] = results_filtered['home_elo'].values
df_features['away_elo'] = results_filtered['away_elo'].values
df_features['elo_diff'] = df_features['home_elo'] - df_features['away_elo']

# Remplir les NaN
df_features['avg_goals_scored'] = df_features['home_goals_scored'].fillna(df_features['home_goals_scored'].mean())
df_features['home_goals_scored'] = df_features['home_goals_scored'].fillna(df_features['home_goals_scored'].mean())
df_features['away_goals_scored'] = df_features['away_goals_scored'].fillna(df_features['away_goals_scored'].mean())
df_features['home_goals_conceded'] = df_features['home_goals_conceded'].fillna(df_features['home_goals_conceded'].mean())
df_features['away_goals_conceded'] = df_features['away_goals_conceded'].fillna(df_features['away_goals_conceded'].mean())

print(df_features.isnull().sum())
print(df_features.shape)

home_win_rate             0
home_goals_scored         0
home_goals_conceded       0
away_win_rate             0
away_goals_scored         0
away_goals_conceded       0
result                    0
home_elo               6679
away_elo               6829
elo_diff               8755
avg_goals_scored          0
dtype: int64
(10132, 11)


In [ ]:
mean_elo = elo_latest['rating'].mean()

df_features['home_elo'] = df_features['home_elo'].fillna(mean_elo)
df_features['away_elo'] = df_features['away_elo'].fillna(mean_elo)
df_features['elo_diff'] = df_features['home_elo'] - df_features['away_elo']

print(df_features.isnull().sum())
print(f"\nMean ELO utilisé : {mean_elo:.0f}")
print(df_features.head())

home_win_rate          0
home_goals_scored      0
home_goals_conceded    0
away_win_rate          0
away_goals_scored      0
away_goals_conceded    0
result                 0
home_elo               0
away_elo               0
elo_diff               0
avg_goals_scored       0
dtype: int64

Mean ELO utilisé : 1779
   home_win_rate  home_goals_scored  home_goals_conceded  away_win_rate  \
0            0.5                1.0                  1.0            0.5   
1            0.5                1.0                  1.0            0.5   
2            0.5                1.0                  1.0            0.5   
3            0.5                1.0                  1.0            0.5   
4            0.5                1.0                  1.0            0.5   

   away_goals_scored  away_goals_conceded    result     home_elo     away_elo  \
0                1.0                  1.0  home_win  1622.000000  1779.083333   
1                1.0                  1.0  home_win  1779.083333  1779.083

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features et target
feature_cols = [
    'home_win_rate', 'home_goals_scored', 'home_goals_conceded',
    'away_win_rate', 'away_goals_scored', 'away_goals_conceded',
    'home_elo', 'away_elo', 'elo_diff'
]

X = df_features[feature_cols]
y = df_features['result']

# Encoder la target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Entraîner
model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

# Evaluer
y_pred = model.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy : 0.597
              precision    recall  f1-score   support

    away_win       0.57      0.58      0.57       608
        draw       0.52      0.14      0.22       449
    home_win       0.62      0.82      0.71       970

    accuracy                           0.60      2027
   macro avg       0.57      0.51      0.50      2027
weighted avg       0.58      0.60      0.56      2027



In [ ]:
# FIFA rank depuis wc_2026_teams
fifa_ranks = teams[['team', 'fifa_rank']].copy()

# Merger pour home team
results_filtered = results_filtered.merge(
    fifa_ranks.rename(columns={'team': 'home_team', 'fifa_rank': 'home_fifa_rank'}),
    on='home_team', how='left'
)

# Merger pour away team
results_filtered = results_filtered.merge(
    fifa_ranks.rename(columns={'team': 'away_team', 'fifa_rank': 'away_fifa_rank'}),
    on='away_team', how='left'
)

# Remplir NaN avec rang moyen
mean_rank = fifa_ranks['fifa_rank'].mean()
results_filtered['home_fifa_rank'] = results_filtered['home_fifa_rank'].fillna(mean_rank)
results_filtered['away_fifa_rank'] = results_filtered['away_fifa_rank'].fillna(mean_rank)

print(results_filtered[['home_team', 'home_fifa_rank', 'away_team', 'away_fifa_rank']].head(10))

      home_team  home_fifa_rank    away_team  away_fifa_rank
0         Egypt         29.0000        Libya         34.0625
1      Cameroon         34.0625       Angola         34.0625
2       Morocco          8.0000  Ivory Coast         33.0000
3          Togo         34.0625     DR Congo         51.0000
4  South Africa         60.0000       Guinea         34.0625
5       Tunisia         40.0000       Zambia         34.0625
6       Nigeria         34.0625        Ghana         65.0000
7      Zimbabwe         34.0625      Senegal         14.0000
8         Egypt         29.0000      Morocco          8.0000
9         Libya         34.0625  Ivory Coast         33.0000


In [ ]:
# Ajouter FIFA rank dans df_features
df_features['home_fifa_rank'] = results_filtered['home_fifa_rank'].values
df_features['away_fifa_rank'] = results_filtered['away_fifa_rank'].values
df_features['fifa_rank_diff'] = df_features['away_fifa_rank'] - df_features['home_fifa_rank']

# Step 2 — Pondération par date (matchs récents = plus de poids)
df_features['date'] = results_filtered['date'].values
df_features['date'] = pd.to_datetime(df_features['date'])

max_date = df_features['date'].max()
df_features['days_ago'] = (max_date - df_features['date']).dt.days
df_features['weight'] = 1 / (1 + df_features['days_ago'] / 365)

print(df_features[['date', 'days_ago', 'weight', 'home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff']].head(10))
print(f"\nWeight min : {df_features['weight'].min():.3f}")
print(f"Weight max : {df_features['weight'].max():.3f}")

        date  days_ago    weight  home_fifa_rank  away_fifa_rank  \
0 2006-01-20      7463  0.046627         29.0000         34.0625   
1 2006-01-21      7462  0.046633         34.0625         34.0625   
2 2006-01-21      7462  0.046633          8.0000         33.0000   
3 2006-01-21      7462  0.046633         34.0625         51.0000   
4 2006-01-22      7461  0.046639         60.0000         34.0625   
5 2006-01-22      7461  0.046639         40.0000         34.0625   
6 2006-01-23      7460  0.046645         34.0625         65.0000   
7 2006-01-23      7460  0.046645         34.0625         14.0000   
8 2006-01-24      7459  0.046651         29.0000          8.0000   
9 2006-01-24      7459  0.046651         34.0625         33.0000   

   fifa_rank_diff  
0          5.0625  
1          0.0000  
2         25.0000  
3         16.9375  
4        -25.9375  
5         -5.9375  
6         30.9375  
7        -20.0625  
8        -21.0000  
9         -1.0625  

Weight min : 0.047
Weight max 

In [ ]:
feature_cols = [
    'home_win_rate', 'home_goals_scored', 'home_goals_conceded',
    'away_win_rate', 'away_goals_scored', 'away_goals_conceded',
    'home_elo', 'away_elo', 'elo_diff',
    'home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff'
]

X = df_features[feature_cols]
y = df_features['result']
weights = df_features['weight']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y_encoded, weights, test_size=0.2, random_state=42
)

model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train, sample_weight=w_train)

y_pred = model.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy : 0.592
              precision    recall  f1-score   support

    away_win       0.56      0.60      0.58       608
        draw       0.38      0.17      0.23       449
    home_win       0.65      0.78      0.71       970

    accuracy                           0.59      2027
   macro avg       0.53      0.52      0.51      2027
weighted avg       0.56      0.59      0.56      2027



In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train, sample_weight=w_train)

print(f"Meilleurs params : {grid_search.best_params_}")
print(f"Meilleur score CV : {grid_search.best_score_:.3f}")

# Evaluer sur test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f"\nAccuracy test : {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Meilleurs params : {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}
Meilleur score CV : 0.583

Accuracy test : 0.598
              precision    recall  f1-score   support

    away_win       0.57      0.59      0.58       608
        draw       0.42      0.17      0.25       449
    home_win       0.64      0.80      0.71       970

    accuracy                           0.60      2027
   macro avg       0.54      0.52      0.51      2027
weighted avg       0.57      0.60      0.57      2027



In [ ]:
tournament_weights = {
    'FIFA World Cup': 3.0,
    'Copa América': 2.5,
    'UEFA Euro': 2.5,
    'African Cup of Nations': 2.5,
    'AFC Asian Cup': 2.5,
    'Gold Cup': 2.0,
    'UEFA Nations League': 2.0,
    'CONCACAF Nations League': 2.0,
    'FIFA World Cup qualification': 1.5,
    'UEFA Euro qualification': 1.5,
    'African Cup of Nations qualification': 1.5,
    'AFC Asian Cup qualification': 1.5,
}

results_filtered['tournament_weight'] = results_filtered['tournament'].map(tournament_weights)

print(results_filtered[['tournament', 'tournament_weight']].drop_duplicates())

                                tournament  tournament_weight
0                   African Cup of Nations                2.5
32             AFC Asian Cup qualification                1.5
52                          FIFA World Cup                3.0
118                UEFA Euro qualification                1.5
131   African Cup of Nations qualification                1.5
404                               Gold Cup                2.0
465                           Copa América                2.5
484                          AFC Asian Cup                2.5
589           FIFA World Cup qualification                1.5
861                              UEFA Euro                2.5
5120                   UEFA Nations League                2.0
5666               CONCACAF Nations League                2.0


In [ ]:
def get_team_stats_v2(team, date, df, n=10):
    matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)

    if len(matches) == 0:
        return {
            'win_rate': 0.5,
            'avg_goals_scored': 1.0,
            'avg_goals_conceded': 1.0
        }

    wins, goals_scored, goals_conceded, total_weight = 0, 0, 0, 0

    for _, row in matches.iterrows():
        w = row['tournament_weight']
        total_weight += w
        if row['home_team'] == team:
            goals_scored += row['home_score'] * w
            goals_conceded += row['away_score'] * w
            if row['result'] == 'home_win':
                wins += w
        else:
            goals_scored += row['away_score'] * w
            goals_conceded += row['home_score'] * w
            if row['result'] == 'away_win':
                wins += w

    return {
        'win_rate': wins / total_weight,
        'avg_goals_scored': goals_scored / total_weight,
        'avg_goals_conceded': goals_conceded / total_weight
    }

def get_h2h(home_team, away_team, date, df):
    matches = df[
        (((df['home_team'] == home_team) & (df['away_team'] == away_team)) |
         ((df['home_team'] == away_team) & (df['away_team'] == home_team))) &
        (df['date'] < date)
    ].tail(10)

    if len(matches) == 0:
        return 0.5

    home_wins = 0
    for _, row in matches.iterrows():
        if (row['home_team'] == home_team and row['result'] == 'home_win') or \
           (row['away_team'] == home_team and row['result'] == 'away_win'):
            home_wins += 1

    return home_wins / len(matches)

# Test
print(get_team_stats_v2('France', '2026-01-01', results_filtered))
print(f"H2H France vs Brazil : {get_h2h('France', 'Brazil', '2026-01-01', results_filtered):.2f}")

{'win_rate': 0.6764705882352942, 'avg_goals_scored': 2.3529411764705883, 'avg_goals_conceded': 1.1764705882352942}
H2H France vs Brazil : 1.00


In [ ]:
features_v2 = []

for _, row in tqdm(results_filtered.iterrows(), total=len(results_filtered)):
    home_stats = get_team_stats_v2(row['home_team'], row['date'], results_filtered)
    away_stats = get_team_stats_v2(row['away_team'], row['date'], results_filtered)
    h2h = get_h2h(row['home_team'], row['away_team'], row['date'], results_filtered)

    features_v2.append({
        'home_win_rate': home_stats['win_rate'],
        'home_goals_scored': home_stats['avg_goals_scored'],
        'home_goals_conceded': home_stats['avg_goals_conceded'],
        'away_win_rate': away_stats['win_rate'],
        'away_goals_scored': away_stats['avg_goals_scored'],
        'away_goals_conceded': away_stats['avg_goals_conceded'],
        'h2h_home_win_rate': h2h,
        'result': row['result']
    })

df_features_v2 = pd.DataFrame(features_v2)
print(df_features_v2.shape)
print(df_features_v2.head())

100%|██████████| 10132/10132 [02:20<00:00, 71.88it/s]

(10132, 8)
   home_win_rate  home_goals_scored  home_goals_conceded  away_win_rate  \
0            0.5                1.0                  1.0            0.5   
1            0.5                1.0                  1.0            0.5   
2            0.5                1.0                  1.0            0.5   
3            0.5                1.0                  1.0            0.5   
4            0.5                1.0                  1.0            0.5   

   away_goals_scored  away_goals_conceded  h2h_home_win_rate    result  
0                1.0                  1.0                0.5  home_win  
1                1.0                  1.0                0.5  home_win  
2                1.0                  1.0                0.5  away_win  
3                1.0                  1.0                0.5  away_win  
4                1.0                  1.0                0.5  away_win  


In [ ]:
# Ajouter ELO et FIFA rank
df_features_v2['home_elo'] = df_features['home_elo'].values
df_features_v2['away_elo'] = df_features['away_elo'].values
df_features_v2['elo_diff'] = df_features_v2['home_elo'] - df_features_v2['away_elo']
df_features_v2['home_fifa_rank'] = df_features['home_fifa_rank'].values
df_features_v2['away_fifa_rank'] = df_features['away_fifa_rank'].values
df_features_v2['fifa_rank_diff'] = df_features_v2['away_fifa_rank'] - df_features_v2['home_fifa_rank']
df_features_v2['weight'] = df_features['weight'].values

# Entraîner
feature_cols_v2 = [
    'home_win_rate', 'home_goals_scored', 'home_goals_conceded',
    'away_win_rate', 'away_goals_scored', 'away_goals_conceded',
    'h2h_home_win_rate', 'home_elo', 'away_elo', 'elo_diff',
    'home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff'
]

X = df_features_v2[feature_cols_v2]
y = le.fit_transform(df_features_v2['result'])

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, df_features_v2['weight'], test_size=0.2, random_state=42
)

model_v2 = XGBClassifier(learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8, random_state=42)
model_v2.fit(X_train, y_train, sample_weight=w_train)

y_pred = model_v2.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy : 0.599
              precision    recall  f1-score   support

    away_win       0.56      0.60      0.58       608
        draw       0.46      0.15      0.23       449
    home_win       0.63      0.81      0.71       970

    accuracy                           0.60      2027
   macro avg       0.55      0.52      0.51      2027
weighted avg       0.57      0.60      0.56      2027



In [ ]:
import joblib

joblib.dump(model_v2, 'model.joblib')
joblib.dump(le, 'label_encoder.joblib')
joblib.dump(feature_cols_v2, 'feature_cols.joblib')

print("Modèle sauvegardé !")
print(f"Classes : {le.classes_}")
print(f"Features : {feature_cols_v2}")

Modèle sauvegardé !
Classes : ['away_win' 'draw' 'home_win']
Features : ['home_win_rate', 'home_goals_scored', 'home_goals_conceded', 'away_win_rate', 'away_goals_scored', 'away_goals_conceded', 'h2h_home_win_rate', 'home_elo', 'away_elo', 'elo_diff', 'home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff']


In [ ]:
# Stats actuelles pour chaque équipe WC 2026
team_stats = {}

for team in wc_teams_official:
    stats = get_team_stats_v2(team, '2026-06-11', results_filtered)
    elo_row = elo_latest[elo_latest['country'] == team]
    elo_val = elo_row['rating'].values[0] if len(elo_row) > 0 else mean_elo

    team_stats[team] = {
        'win_rate': stats['win_rate'],
        'avg_goals_scored': stats['avg_goals_scored'],
        'avg_goals_conceded': stats['avg_goals_conceded'],
        'elo': elo_val
    }

print(f"Equipes préparées : {len(team_stats)}")
print(team_stats['France'])
print(team_stats['Argentina'])

Equipes préparées : 48
{'win_rate': 0.6764705882352942, 'avg_goals_scored': 2.3529411764705883, 'avg_goals_conceded': 1.1764705882352942, 'elo': np.int64(2062)}
{'win_rate': 0.6, 'avg_goals_scored': 1.9, 'avg_goals_conceded': 0.6, 'elo': np.int64(2113)}


In [ ]:
import numpy as np
import random

def predict_match(home_team, away_team, team_stats, fifa_ranks, model, le, feature_cols):
    home = team_stats[home_team]
    away = team_stats[away_team]

    home_rank = fifa_ranks.get(home_team, mean_rank)
    away_rank = fifa_ranks.get(away_team, mean_rank)
    h2h = get_h2h(home_team, away_team, '2026-06-11', results_filtered)

    features = pd.DataFrame([{
        'home_win_rate': home['win_rate'],
        'home_goals_scored': home['avg_goals_scored'],
        'home_goals_conceded': home['avg_goals_conceded'],
        'away_win_rate': away['win_rate'],
        'away_goals_scored': away['avg_goals_scored'],
        'away_goals_conceded': away['avg_goals_conceded'],
        'h2h_home_win_rate': h2h,
        'home_elo': home['elo'],
        'away_elo': away['elo'],
        'elo_diff': home['elo'] - away['elo'],
        'home_fifa_rank': home_rank,
        'away_fifa_rank': away_rank,
        'fifa_rank_diff': away_rank - home_rank
    }])

    probs = model.predict_proba(features[feature_cols])[0]
    classes = le.classes_

    prob_dict = dict(zip(classes, probs))

    # Tirer un résultat aléatoire basé sur les probabilités
    outcome = random.choices(
        ['away_win', 'draw', 'home_win'],
        weights=[prob_dict['away_win'], prob_dict['draw'], prob_dict['home_win']]
    )[0]

    if outcome == 'home_win':
        return home_team
    elif outcome == 'away_win':
        return away_team
    else:
        # En cas de draw en knockout → tirage aléatoire 50/50
        return random.choice([home_team, away_team])

# FIFA ranks dict
fifa_ranks_dict = dict(zip(teams['team'], teams['fifa_rank']))

# Test
winner = predict_match('France', 'Brazil', team_stats, fifa_ranks_dict, model_v2, le, feature_cols_v2)
print(f"Vainqueur prédit : {winner}")

Vainqueur prédit : France


In [ ]:
def simulate_tournament(fixtures, team_stats, fifa_ranks_dict, model, le, feature_cols):
    # Phase de groupes
    group_points = {team: 0 for team in wc_teams_official}
    group_gd = {team: 0 for team in wc_teams_official}

    group_matches = fixtures[fixtures['stage'] == 'Group Stage']

    for _, match in group_matches.iterrows():
        home = match['team1']
        away = match['team2']

        if home not in team_stats or away not in team_stats:
            continue

        home_s = team_stats[home]
        away_s = team_stats[away]
        h2h = get_h2h(home, away, '2026-06-11', results_filtered)

        features = pd.DataFrame([{
            'home_win_rate': home_s['win_rate'],
            'home_goals_scored': home_s['avg_goals_scored'],
            'home_goals_conceded': home_s['avg_goals_conceded'],
            'away_win_rate': away_s['win_rate'],
            'away_goals_scored': away_s['avg_goals_scored'],
            'away_goals_conceded': away_s['avg_goals_conceded'],
            'h2h_home_win_rate': h2h,
            'home_elo': home_s['elo'],
            'away_elo': away_s['elo'],
            'elo_diff': home_s['elo'] - away_s['elo'],
            'home_fifa_rank': fifa_ranks_dict.get(home, mean_rank),
            'away_fifa_rank': fifa_ranks_dict.get(away, mean_rank),
            'fifa_rank_diff': fifa_ranks_dict.get(away, mean_rank) - fifa_ranks_dict.get(home, mean_rank)
        }])

        probs = model.predict_proba(features[feature_cols])[0]
        classes = le.classes_
        prob_dict = dict(zip(classes, probs))

        outcome = random.choices(
            ['away_win', 'draw', 'home_win'],
            weights=[prob_dict['away_win'], prob_dict['draw'], prob_dict['home_win']]
        )[0]

        if outcome == 'home_win':
            group_points[home] += 3
            group_gd[home] += 1
        elif outcome == 'away_win':
            group_points[away] += 3
            group_gd[away] += 1
        else:
            group_points[home] += 1
            group_points[away] += 1

    # Qualifier top 2 par groupe
    groups = fixtures[fixtures['stage'] == 'Group Stage']['group'].unique()
    qualified = []

    for group in groups:
        group_teams = fixtures[
            (fixtures['stage'] == 'Group Stage') &
            (fixtures['group'] == group)
        ][['team1', 'team2']].values.flatten()
        group_teams = list(set(group_teams))
        group_teams = [t for t in group_teams if t in team_stats]

        group_teams.sort(key=lambda t: (group_points[t], group_gd[t]), reverse=True)
        qualified.extend(group_teams[:2])

    # Phase knockout
    knockout_teams = qualified.copy()

    while len(knockout_teams) > 1:
        next_round = []
        random.shuffle(knockout_teams)
        for i in range(0, len(knockout_teams), 2):
            if i+1 < len(knockout_teams):
                winner = predict_match(
                    knockout_teams[i], knockout_teams[i+1],
                    team_stats, fifa_ranks_dict, model, le, feature_cols
                )
                next_round.append(winner)
        knockout_teams = next_round

    return knockout_teams[0] if knockout_teams else None

# Test une simulation
winner = simulate_tournament(fixtures, team_stats, fifa_ranks_dict, model_v2, le, feature_cols_v2)
print(f"Vainqueur simulé : {winner}")

Vainqueur simulé : Norway


In [ ]:
from collections import Counter

N_SIMULATIONS = 10000
winners = []

for _ in tqdm(range(N_SIMULATIONS)):
    winner = simulate_tournament(fixtures, team_stats, fifa_ranks_dict, model_v2, le, feature_cols_v2)
    if winner:
        winners.append(winner)

# Calculer les probabilités
win_counts = Counter(winners)
win_probs = {team: count/N_SIMULATIONS*100 for team, count in win_counts.items()}
win_probs = dict(sorted(win_probs.items(), key=lambda x: x[1], reverse=True))

print("\n🏆 TOP 10 FAVORIS WC 2026 :")
for i, (team, prob) in enumerate(list(win_probs.items())[:10], 1):
    print(f"{i}. {team}: {prob:.1f}%")

100%|██████████| 10000/10000 [3:13:59<00:00,  1.16s/it]


🏆 TOP 10 FAVORIS WC 2026 :
1. Croatia: 7.5%
2. Argentina: 7.2%
3. Morocco: 6.7%
4. Mexico: 5.6%
5. France: 5.3%
6. England: 4.8%
7. Japan: 4.2%
8. Brazil: 3.5%
9. Netherlands: 3.1%
10. Spain: 3.1%
